# Библиотеки

In [2]:
from itertools import combinations

import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg
import seaborn as sns

## 10.9 Анализ надежности-согласованности

Это данные опроса операторов CallCenter одной из компаний по удовлетворенности их работой в целом и отдельными ее аспектами. Оценка по 5-бальной шкале:  1 Абсолютно не удовлетворен\не согласен и 5 Полностью удовлетворен\согласен.

Кронбах альфа это показатель внутренней согласованности набора вопросов, то есть насколько они измеряют одну и ту же скрытую штуку (конструкт) и дают похожий сигнал.

Интуитивно: если вопросы про одно и то же, ответы по ним будут коррелировать, и альфа будет выше. Если вопросы про разное или часть вопросов шумная, альфа ниже.

In [17]:
d = pd.read_excel('./data/SPSS28.xlsx')

In [18]:
d.dtypes

РеализацияСпособностей        int64
СодержаниеРаботы              int64
УсловияРаботы                 int64
ПрофРост                      int64
ДостаточностьЗнанийНавыков    int64
КомфортностьУсловий           int64
Зарплата                      int64
ЛинейныйРуководитель          int64
УдовРаботойвЦелом             int64
dtype: object

In [19]:
d.replace(-99, pd.NA, inplace=True)

In [21]:
d = d.astype('Int64')

In [35]:
d.columns

Index(['РеализацияСпособностей', 'СодержаниеРаботы', 'УсловияРаботы',
       'ПрофРост', 'ДостаточностьЗнанийНавыков', 'КомфортностьУсловий',
       'Зарплата', 'ЛинейныйРуководитель', 'УдовРаботойвЦелом'],
      dtype='object')

In [23]:
pg.cronbach_alpha(d)

(np.float64(0.7675879170018916), array([0.742, 0.792]))

In [49]:
alpha_cron = pg.cronbach_alpha(d)[0]

for col in d.columns:
    sub_n = d.drop(columns=[col])
    alphaD, l = pg.cronbach_alpha(sub_n)
    print(f'Альфа без {col:<27}: is_upper:{int(alpha_cron < alphaD)}, is_sign:{int(alpha_cron < l[0])}, {alphaD:g} {l}')

Альфа без РеализацияСпособностей     : is_upper:0, is_sign:0, 0.715932 [0.684 0.746]
Альфа без СодержаниеРаботы           : is_upper:0, is_sign:0, 0.733811 [0.704 0.762]
Альфа без УсловияРаботы              : is_upper:0, is_sign:0, 0.726628 [0.696 0.755]
Альфа без ПрофРост                   : is_upper:0, is_sign:0, 0.733316 [0.703 0.761]
Альфа без ДостаточностьЗнанийНавыков : is_upper:1, is_sign:1, 0.794194 [0.771 0.816]
Альфа без КомфортностьУсловий        : is_upper:1, is_sign:0, 0.768251 [0.742 0.793]
Альфа без Зарплата                   : is_upper:0, is_sign:0, 0.7429 [0.714 0.77 ]
Альфа без ЛинейныйРуководитель       : is_upper:0, is_sign:0, 0.756741 [0.729 0.782]
Альфа без УдовРаботойвЦелом          : is_upper:0, is_sign:0, 0.725625 [0.695 0.755]


In [41]:
pg.cronbach_alpha(
    d.drop(columns=['ДостаточностьЗнанийНавыков', 'КомфортностьУсловий'])
)

(np.float64(0.8059942304555361), array([0.784, 0.827]))

In [36]:
best_alpha = -1
best_cols = None

cols = d.columns
for k in range(2, len(cols)+1):
    for comb in combinations(cols, k):
        a = pg.cronbach_alpha(d[list(comb)])[0]
        if a > best_alpha:
            best_alpha = a
            best_cols = comb

best_alpha, best_cols

(np.float64(0.8095968384909097),
 ('РеализацияСпособностей',
  'СодержаниеРаботы',
  'УсловияРаботы',
  'ПрофРост',
  'Зарплата',
  'УдовРаботойвЦелом'))